# Digital Brain pilot — website to PGL
Run on the lab Mac with Python 3.12+, the pinned PGL experiment extra, its native prerequisites, and FFmpeg installed. No paid server is required. This notebook runs the actual Digital Brain task with website-assigned local videos and saves native PGL output plus a durable trial journal.

**Non-participant integration pilot only.** Display, response devices, eye tracker, timing, and the scientific new/old/foil schedule still need lab validation. The package deliberately remains `pgl_ready: false`.

In the updated website, save a study, open its Study integration section, publish an integration test, and choose **Pair workstation**. Use the local website origin or a live HTTPS origin running the same server integration. The old live website cannot supply these APIs until its code is updated.

In [ ]:
from dbp_pgl_runner.runner import StudyRunner
from dbp_pgl_runner.pgl_adapter import RunSettings
runner = StudyRunner()
subject = 's001'

## Pair once
Skip this cell if already paired. The one-time code is hidden, never placed in a URL or shell history. Private-file storage is explicit; the credential is stored locally with mode 0600 inside a private 0700 directory. Do not save secrets in notebook cells or outputs.

In [ ]:
from getpass import getpass
import socket
origin = input('Website origin (HTTPS or local loopback): ').strip()
runner.connect(origin, getpass('One-time pairing code: '), socket.gethostname(), allow_file_token=True)

## Prepare and inspect
The server supplies the exact study subject and ordered parent/repeat/foil media. Downloads resume safely and hashes are checked. A foil is its assigned rendered interval, never the full parent. FFmpeg and ffprobe must be configured on the server for studies containing foils.

In [ ]:
prepared = runner.prepare(subject)
runner.status(subject)

## Run the actual PGL pilot
This next cell opens the experiment display and uses your installed PGL configuration. Set the lab's saved settings/display names below as appropriate. It fully decodes videos first, reserves an exclusive attempt, and then presents without network requests. Native results and journal events are saved locally; synchronization happens after presentation. Day/block values identify this integration run, not an approved experimental allocation.

Do not rerun this cell to retry an upload. An interrupted or already-run attempt is never silently replayed.

In [ ]:
settings = RunSettings(day=1, block=1, description_seconds=12, display_width=50)
result = runner.run(subject, integration_test=True, settings=settings)
result

## Results and retrying synchronization
The result reports the private `attempt_root`, completion and synchronization separately. Native files live under its `native/` directory; `events.jsonl` is the hashed, ordered trial journal. The website's **PGL runs** panel shows received attempts when refreshed. This cell can be repeated safely; it does not show videos again.

In [ ]:
runner.sync(subject)
runner.status(subject)

## If the process or notebook crashed
Inspect `runner.status(subject)` first. If an attempt remains unfinished, use `runner.recover(subject, terminate=True)` and then `runner.sync(subject)`. Only when the error specifically identifies an incomplete final journal fragment, explicitly add `repair_tail=True`. Recovery preserves completed-trial evidence and ends the attempt; it does not resume or silently replay exposed trials. The first version intentionally requires a new, reviewed study assignment after exposure rather than automatically creating a replacement presentation.

Keep this notebook output-free when committing or sharing it. Native participant responses, credentials, media, and runtime journals never belong in the source repository.